In [1]:
# Clone Real-ESRGAN and enter the Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr
!pip install facexlib
!pip install gfpgan

# colab 本身就有裝這些 排除以下的套件
!sed -i '/opencv-python/d' requirements.txt
!sed -i '/Pillow/d' requirements.txt
!sed -i '/torch>=1.7/d' requirements.txt
!sed -i '/torchvision/d' requirements.txt
!sed -i '/tqdm/d' requirements.txt



!pip install -r requirements.txt
!python setup.py develop

# 下載漫畫文字segmentation模型的權重
!pip install ultralytics pillow opencv-python
!wget -O best.pt "https://huggingface.co/ShadowB/Manga109-panel-balloon-text-yolov26-segmentation/resolve/main/best.pt"


# 下載漫畫線搞提取器的模型權重
!git clone https://github.com/ljsabc/MangaLineExtraction_PyTorch.git
!wget -O /content/Real-ESRGAN/MangaLineExtraction_PyTorch/erika.pth https://github.com/ljsabc/MangaLineExtraction_PyTorch/releases/download/v1/erika.pth






Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Total 759 (delta 0), reused 0 (delta 0), pack-reused 759 (from 1)
Receiving objects: 100% (759/759), 5.39 MiB | 34.47 MiB/s, done.
Resolving deltas: 100% (408/408), done.
/content/Real-ESRGAN
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 11.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.7/344.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 137.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 27.6 MB/s eta 0:00:00
  Created wheel for basicsr: filename=basicsr-1.4.2-py3-none-any.whl size=214816 sha256=e7b0782fecabc82b39c1ceac602eafaa4d90cb65e5b8de6d20b1927aa68b4eda
  Stored in directory: /root/.cache/pip/wheels/9a/e3/e4/58f29bfabb622dd40b6d9839318ce5bf092062b81ca3aa19ea
Successfully built basicsr
 

In [2]:
import os
import zipfile
import shutil
import subprocess

import cv2
import numpy as np
import torch
import skimage

from tqdm import tqdm
from ultralytics import YOLO
from natsort import natsorted

from tqdm import tqdm
from MangaLineExtraction_PyTorch.model_torch import res_skip

line_extraction_model = res_skip()
line_extraction_model.load_state_dict(torch.load(r'/content/Real-ESRGAN/MangaLineExtraction_PyTorch/erika.pth'))

line_extraction_model.cuda()
line_extraction_model.eval()

print("Setup Complete")


def line_extraction(model, image_path):
  """提取線稿 回傳灰階線稿"""

  with torch.no_grad():

      src = cv2.imdecode(np.fromfile(file=image_path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)

      rows = int(np.ceil(src.shape[0]/16))*16
      cols = int(np.ceil(src.shape[1]/16))*16

      # manually construct a batch. You can change it based on your usecases.
      patch = np.ones((1,1,rows,cols),dtype="float32")
      patch[0,0,0:src.shape[0],0:src.shape[1]] = src

      tensor = torch.from_numpy(patch).cuda()
      y = model(tensor)

      yc = y.cpu().numpy()[0,0,:,:]
      yc[yc>255] = 255
      yc[yc<0] = 0


      output = yc[0:src.shape[0],0:src.shape[1]]

  return output





def yolo_inference(model, image_path):
    """回傳文字區域mask"""
    results = model.predict(
        source=image_path,
        imgsz=1920,
        conf=0.25,
        iou=0.7,
        retina_masks=True,
        verbose=False
    )

    for result in results:
        if result.masks is None:
            print(f"{image_path}:\n找不到文字\n")
            h, w = result.orig_shape
            text_mask = np.zeros((h, w), dtype=np.uint8)
            return text_mask


        # 建立黑底 mask
        h, w = result.orig_shape
        text_mask = np.zeros((h, w), dtype=np.uint8)

        for i, cls in enumerate(result.boxes.cls):
            class_id = int(cls)

            # 只保留 text
            if class_id == 1:
                mask = result.masks.data[i].cpu().numpy()

                # mask resize 到原圖大小
                mask = cv2.resize(
                    mask,
                    (w, h),
                    interpolation=cv2.INTER_NEAREST
                )

                # 疊加多個 text mask
                text_mask[mask > 0.5] = 255

    return text_mask



def blur(img, blur_amount=5):
    '''
    此方法來自 https://github.com/natethegreate/Screentone-Remover

    藉由模糊來去除網點
    '''

    if(blur_amount == 7):
        dst2 = cv2.GaussianBlur(img,(7,7),0)
        dst = cv2.bilateralFilter(dst2, 7, 80, 80)
    else:
        dst2 = cv2.GaussianBlur(img,(5,5),0)
        dst = cv2.bilateralFilter(dst2, 7, 10 * blur_amount, 80)

    return dst








ROOT = r'/content'

book_LUT = {} # {book_0 : 書名, book_1 : 書名}

valid_ext = [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"]

RAW_BOOK_DIR = os.path.join(ROOT, 'raw_books')
SOURCE_DIR = os.path.join(ROOT, 'sources')
RESULT_DIR = os.path.join(ROOT, 'results')
DRAFT_DIR = os.path.join(ROOT, 'drafts')
UPSCALE_DIR = os.path.join(ROOT, 'upscale')
SCREENTONE_REMOVE_DIR = os.path.join(ROOT, 'screentone_remove')
DETAIL_ENHANCE_DIR = os.path.join(ROOT, 'detail_enhance')

# 初始化清空
if os.path.exists(DETAIL_ENHANCE_DIR):
    shutil.rmtree(DETAIL_ENHANCE_DIR)


if os.path.exists(RAW_BOOK_DIR):
    shutil.rmtree(RAW_BOOK_DIR)

if os.path.exists(SOURCE_DIR):
    shutil.rmtree(SOURCE_DIR)

if os.path.exists(RESULT_DIR):
    shutil.rmtree(RESULT_DIR)

if os.path.exists(DRAFT_DIR):
    shutil.rmtree(DRAFT_DIR)

if os.path.exists(SCREENTONE_REMOVE_DIR):
    shutil.rmtree(SCREENTONE_REMOVE_DIR)

if os.path.exists(UPSCALE_DIR):
    shutil.rmtree(UPSCALE_DIR)


# 建立資料夾
if not os.path.exists(RAW_BOOK_DIR):
    os.makedirs(RAW_BOOK_DIR)

if not os.path.exists(SOURCE_DIR):
    os.makedirs(SOURCE_DIR)

if not os.path.exists(RESULT_DIR):
    os.makedirs(RESULT_DIR)

if not os.path.exists(DRAFT_DIR):
    os.makedirs(DRAFT_DIR)

if not os.path.exists(SCREENTONE_REMOVE_DIR):
    os.makedirs(SCREENTONE_REMOVE_DIR)

if not os.path.exists(UPSCALE_DIR):
    os.makedirs(UPSCALE_DIR)

if not os.path.exists(DETAIL_ENHANCE_DIR):
    os.makedirs(DETAIL_ENHANCE_DIR)


# 取得書名 (不取後面的附檔名)
book_names = [os.path.splitext(name)[0] for name in os.listdir(ROOT) if name.endswith('.zip')]


# 解壓縮
for i, book_name in enumerate(book_names):
    book_LUT[f"book_{i}"] = book_name

    src = os.path.join(ROOT, f'{book_name}.zip')
    trg = os.path.join(RAW_BOOK_DIR, book_name)

    # 解壓縮後會變成這樣 raw_books\他的書名\他的書名  因為我們是壓縮整個資料夾
    with zipfile.ZipFile(src, 'r') as zip_ref:
        zip_ref.extractall(trg)


# 建 SOURCE_DIR 裡的資料夾
for book_name in book_names:
    os.makedirs(os.path.join(SOURCE_DIR, book_name))


# 建 SCREENTONE_REMOVE_DIR 裡的資料夾
for book_name in book_names:
    os.makedirs(os.path.join(SCREENTONE_REMOVE_DIR, book_name))

# 建 DRAFT_DIR 裡的資料夾
for book_name in book_names:
    os.makedirs(os.path.join(DRAFT_DIR, book_name))

# 建 DETAIL_ENHANCE_DIR 裡的資料夾
for book_name in book_names:
    os.makedirs(os.path.join(DETAIL_ENHANCE_DIR, book_name))


# 把相片移動到 SOURCE_DIR 只複製相片防止資料夾裡有其他東西
for book_name in book_names:

    # raw_books\他的書名\他的書名
    book_dir = os.path.join(RAW_BOOK_DIR, book_name, book_name)

    # 處理每個資料夾 是相片就複製起來
    for file in os.listdir(book_dir):

        file_name, ext = os.path.splitext(file)

        # 是相片就複製起來
        if ext.lower() in valid_ext:
            src = os.path.join(book_dir,file)
            trg = os.path.join(SOURCE_DIR, book_name, file)
            shutil.copy(src, trg)



# 載入模型
text_model = YOLO("/content/Real-ESRGAN/best.pt")


# 去除網點與取出文字區域
print("去除網點中...")
for book_name in tqdm(book_names):

    # 讀取相片位置
    image_dir = os.path.join(SOURCE_DIR, book_name)
    image_paths = [os.path.join(image_dir,file_name) for file_name in os.listdir(image_dir)]



    # 針對每一張圖片進行去除網點與取出文字區域
    for image_path in image_paths:
        image = cv2.imdecode(np.fromfile(file=image_path, dtype=np.uint8), cv2.IMREAD_COLOR)

        # 使用模糊來消除網點 要根據網點的大小與相片大小來決定 小顆的用3 中的用5 大的用7
        blur_image = blur(image, 5)

        # 使用yolo提取文字區域 (文字切得不好可以調dilate參數來調整區域範圍)
        text_mask = yolo_inference(text_model, image_path)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        text_mask = cv2.dilate(text_mask, kernel, iterations=0)

        # mask 二值化
        text_mask = text_mask > 0

        # 轉成跟原圖一樣變成三通道
        text_mask = np.stack([text_mask]*3, axis=-1)

        # 文字範圍就用原圖，不然就用 blur_image
        blend = np.where(text_mask, image, blur_image)


        # 存檔
        base_name = os.path.basename(image_path)
        _, ext = os.path.splitext(base_name)
        trg = os.path.join(SCREENTONE_REMOVE_DIR, book_name, base_name)
        cv2.imencode(ext, blend)[1].tofile(trg)



# 重新命名資料夾 因為REAL-ESRGAN 不能吃中文
for book_key, book_name in book_LUT.items():  # {book_0 : ??W, book_1 : ??W}
    old_name = os.path.join(SCREENTONE_REMOVE_DIR, book_name)
    new_name = os.path.join(SCREENTONE_REMOVE_DIR, book_key)
    os.rename(old_name, new_name)


# 改掉第三方模組的東西 新版的改方法名稱了
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.10/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.11/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /usr/local/lib/python3.13/dist-packages/basicsr/data/degradations.py





print("圖片修復中...")
for book_key, book_name in tqdm(book_LUT.items()):

    input_dir = os.path.join(SCREENTONE_REMOVE_DIR,book_key)
    output_dir = os.path.join(UPSCALE_DIR,book_key)
    subprocess.run([
        "python",
        "/content/Real-ESRGAN/inference_realesrgan.py",
        "-i", input_dir,
        "-n", "realesr-general-x4v3",
        "-s", "1",
        "-o", output_dir,
        "--denoise_strength", "0.4"
    ], check=True)





# 把細節補回來
print("把圖片細節補回來中...")
for book_key, book_name in tqdm(book_LUT.items()):

    # 讀取相片位置
    image_dir = os.path.join(SOURCE_DIR, book_name)
    image_paths = [os.path.join(image_dir,file_name) for file_name in os.listdir(image_dir)]


    upscale_dir = os.path.join(UPSCALE_DIR, book_key)
    upscale_paths = [os.path.join(upscale_dir,file_name) for file_name in os.listdir(upscale_dir)]

    # 整理順序 這樣 zip 才會對上
    image_paths = natsorted(image_paths)
    upscale_paths = natsorted(upscale_paths)


    # 針對每一張圖片進行去除網點與取出文字區域
    for image_path, upscale_path in zip(image_paths,upscale_paths):

        image = cv2.imdecode(np.fromfile(file=image_path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)

        upscale_image = cv2.imdecode(np.fromfile(file=upscale_path, dtype=np.uint8), cv2.IMREAD_COLOR)

        upscale_lab = cv2.cvtColor(upscale_image, cv2.COLOR_BGR2LAB)
        L, A, B = cv2.split(upscale_lab)

        draft_image = line_extraction(line_extraction_model, image_path)
        draft_image = np.clip(draft_image, 0, 255).astype(np.uint8) # 0~1 轉 0~255

        draft_inverse = cv2.bitwise_not(draft_image)

        after_gamma = skimage.exposure.adjust_gamma(draft_inverse, gamma=0.5, gain=0.2)

        after_gamma = np.clip(after_gamma, 0, 255).astype(np.uint8)


        L = cv2.subtract(L, after_gamma)

        lab = cv2.merge([L, A, B])

        enhance = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

        # ehance = cv2.subtract(upscale_image, after_gamma)


        # 存檔
        base_name = os.path.basename(image_path)
        file_head, _ = os.path.splitext(base_name)
        output_path = os.path.join(DETAIL_ENHANCE_DIR,book_name,f"{file_head}.png")
        cv2.imencode(".png", enhance)[1].tofile(output_path)



    # 壓縮資料夾
    src_dir = os.path.join(DETAIL_ENHANCE_DIR, book_name)
    out_put = os.path.join(RESULT_DIR, book_name)

    shutil.make_archive(
        out_put,   # 輸出檔名，不用加 .zip
        "zip",     # 輸出格式
        src_dir   # 要壓縮的資料夾
    )


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Setup Complete
去除網點中...


  0%|          | 0/2 [00:00<?, ?it/s]

/content/sources/toy_data_grayscale/02.jpg:
找不到文字



 50%|█████     | 1/2 [00:16<00:16, 17.00s/it]

/content/sources/toy_data_color/02.jpg:
找不到文字

/content/sources/toy_data_color/49.jpg:
找不到文字

/content/sources/toy_data_color/50.jpg:
找不到文字



...

sed: can't read /usr/local/lib/python3.10/dist-packages/basicsr/data/degradations.py: No such file or directory


sed: can't read /usr/local/lib/python3.11/dist-packages/basicsr/data/degradations.py: No such file or directory
sed: can't read /usr/local/lib/python3.13/dist-packages/basicsr/data/degradations.py: No such file or directory
圖片修復中...


100%|██████████| 2/2 [05:08<00:00, 154.05s/it]


把圖片細節補回來中...


100%|██████████| 2/2 [02:59<00:00, 89.70s/it]
